# 02 — Preprocessing experiments
We preserve a held-out test set. Imputation and scaling are fitted on training data only. Invalid zeros are converted to missing values only in the five physiologically implausible measurement columns.

In [ ]:
%pip -q install pandas numpy scikit-learn

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
DATA_PATH=Path("../dataset/diabetes.csv")
if not DATA_PATH.exists(): DATA_PATH=Path("/content/diabetes.csv")
df=pd.read_csv(DATA_PATH)
TARGET="Outcome"
X=df.drop(columns=TARGET).copy()
y=df[TARGET].astype(int).copy()
INVALID_ZERO_COLS=["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
X[INVALID_ZERO_COLS]=X[INVALID_ZERO_COLS].replace(0,np.nan)
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,random_state=42,stratify=y)
print("Train:",X_train.shape,"Test:",X_test.shape)
print("Train outcome proportions:",y_train.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Fit each imputer on training data only; transform the held-out test data.
imputers={
 "median": SimpleImputer(strategy="median"),
 "knn": KNNImputer(n_neighbors=5, weights="distance"),
 "iterative": IterativeImputer(random_state=42,max_iter=10)
}
prepared={}
for name, imputer in imputers.items():
    Xt=imputer.fit_transform(X_train)
    Xv=imputer.transform(X_test)
    scaler=StandardScaler()
    Xt=scaler.fit_transform(Xt)
    Xv=scaler.transform(Xv)
    prepared[name]={"X_train":Xt,"X_test":Xv,"imputer":imputer,"scaler":scaler}
    print(name, Xt.shape, Xv.shape, "finite:",np.isfinite(Xt).all() and np.isfinite(Xv).all())

In [ ]:
# Save split and fitted preprocessing artifacts for reproducibility.
import joblib, json
from pathlib import Path
out=Path("../models/preprocessing")
if not out.exists(): out=Path("models/preprocessing")
out.mkdir(parents=True,exist_ok=True)
for name,obj in prepared.items():
    joblib.dump(obj["imputer"],out/f"{name}_imputer.joblib")
    joblib.dump(obj["scaler"],out/f"{name}_scaler.joblib")
np.savez(out/"train_test_split.npz",
    X_train=X_train.to_numpy(),X_test=X_test.to_numpy(),
    y_train=y_train.to_numpy(),y_test=y_test.to_numpy())
print("Saved preprocessing artifacts to",out.resolve())

## Important
This notebook creates a fixed stratified 80/20 holdout for the initial experiments. For final model selection, all preprocessing and feature selection must also be inside cross-validation folds (for example, via sklearn Pipelines). Do not tune decisions using the test set.